In [2]:
# ----------------- Step 0: Install Required Packages -----------------
!pip install -q pandas pyarrow sentence-transformers faiss-cpu chromadb openai


[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\manjo\Downloads\wiki_hybrid_chunks_300 (1).parquet")
print("Available columns:\n", df.columns)

Available columns:
 Index(['group_id', 'section', 'chunk_id', 'chunk_text'], dtype='object')


In [4]:
# Use the correct text column
df['text'] = df['chunk_text'].astype(str)
texts = df['text'].tolist()

In [7]:
# -------------------- Step 2: Generate Embeddings --------------------
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True)


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [8]:
# -------------------- Step 3: Save Embeddings to New Parquet --------------------
import numpy as np

embedding_df = pd.DataFrame(embeddings, columns=[f"emb_{i}" for i in range(len(embeddings[0]))])
df_with_embeddings = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)
df_with_embeddings.to_parquet("wiki_chunks_with_embeddings.parquet")
print("✅ Saved: wiki_chunks_with_embeddings.parquet")

✅ Saved: wiki_chunks_with_embeddings.parquet


In [9]:
# -------------------- Step 4A: Store in FAISS --------------------
import faiss

embedding_array = np.array(embeddings).astype("float32")
faiss_index = faiss.IndexFlatL2(embedding_array.shape[1])
faiss_index.add(embedding_array)
faiss.write_index(faiss_index, "wiki_chunks_faiss.index")
print("✅ Saved FAISS index")

✅ Saved FAISS index


In [10]:
# --------------------- Step 5: Retrieve Relevant Chunks ---------------------

from sentence_transformers import SentenceTransformer

# Use a separate variable to avoid conflict with TinyLlama later
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Input user query
query = "What is the use of reinforcement learning in chatbots?"

# Encode the query using the sentence-transformer embedding model (not TinyLlama)
query_embedding = embedder.encode([query]).astype("float32")

# Search in FAISS index to get top-3 most relevant chunks (top-k = 3)
D, I = faiss_index.search(query_embedding, k=3)

# Retrieve text chunks using indices returned by FAISS
retrieved_chunks = [texts[i] for i in I[0]]

# Join the chunks into a single string to pass to TinyLlama
retrieved_context = " ".join(retrieved_chunks)

# Print the retrieved context for verification
print("📚 Retrieved Context:\n", retrieved_context)

📚 Retrieved Context:
 about how they spend or invest money. a player s strategy usually revolves around brass franchise games the business game terra mystica franchise games great western trail catan food chain magnate the game of life monopoly power grid ensuring they have enough resources to achieve a strong financial position. economic board games usually simulate a market in some way. these games are often also called economic simulation games. the term economic board game is often used interchangeably with resource management board game. terraforming mars through the ages twilight imperium educational educational board games are those designed to teach new ideas, concepts, topics, and or understanding, while playing. the board game s learning is based ona particular theme. while educational games exist for different age groups, they are usually designed for children. the magic labyrinth brain quest cashflow evolution franchise games mariposas wingspan engine builder engine builder

In [11]:
# --------------------- Step 6: Build Prompt from Retrieved Chunks ---------------------

# Define a function that creates a prompt for TinyLlama to generate a proper answer
def build_prompt(context, question):
    return f"""You are a helpful assistant. Only use the context below to answer the user's question clearly and concisely.

Context:
{context}

Question: {question}
Answer:"""

In [ ]:
# Generate Answer Using Local LLM

In [26]:
pip install transformers accelerate torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Step: Load TinyLlama and Generate Answer from Retrieved Context
# In this step, I load the TinyLlama model and use it to generate answers 
# based on the context retrieved from FAISS using the user query.

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

In [13]:
def build_prompt(context, question):
    return f"""You are a helpful assistant. Only use the context below to answer the user's question clearly and concisely.

Context:
{context}

Question: {question}
Answer:"""

# Use previously retrieved context and query
prompt = build_prompt(retrieved_context, query)

In [14]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    output = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    return tokenizer.decode(output[0], skip_special_tokens=True)

answer = generate_answer(prompt)
print("🤖 Final Answer from TinyLlama:\n", answer)

C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


🤖 Final Answer from TinyLlama:
 You are a helpful assistant. Only use the context below to answer the user's question clearly and concisely.

Context:
about how they spend or invest money. a player s strategy usually revolves around brass franchise games the business game terra mystica franchise games great western trail catan food chain magnate the game of life monopoly power grid ensuring they have enough resources to achieve a strong financial position. economic board games usually simulate a market in some way. these games are often also called economic simulation games. the term economic board game is often used interchangeably with resource management board game. terraforming mars through the ages twilight imperium educational educational board games are those designed to teach new ideas, concepts, topics, and or understanding, while playing. the board game s learning is based ona particular theme. while educational games exist for different age groups, they are usually designed 

In [ ]:
#  Ask General Questions Using TinyLlama Without Any Context (No RAG)

In [15]:
# Ask anything directly (no context)
question = "Why do stars twinkle at night?"
prompt = f"Question: {question}\nAnswer:"

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    output = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Get and print answer
answer = generate_answer(prompt)
print("🤖 TinyLlama Answer:\n", answer)

🤖 TinyLlama Answer:
 Question: Why do stars twinkle at night?
Answer: Stars twinkle at night because they are made of gas and dust, which glows when heated by the sun. The gas and dust are heated by the sun's radiation, which causes the gas and dust to vibrate and emit light. The vibrations are amplified by the gas and dust's density, which causes the light to appear as a bright, star-like object.


In [ ]:
# Step: Enhance Reliability with Cache  & Fallback  Logic for TinyLlama Responses
# This block checks if the answer already exists in cache (for faster response).
# If not, it tries to generate a new response using TinyLlama.
# If generation fails or response is empty, a fallback response is used to ensure a smooth user experience.

In [17]:
from diskcache import Cache
import random

# Step 1: Initialize cache
cache = Cache("./llm_cache")

# Step 2: Define fallback responses
fallback_responses = [
    "I'm sorry, I don't have a good answer for that.",
    "Unfortunately, I couldn't find enough information to answer your question.",
    "I'm unable to answer that at the moment. Please try rephrasing your question.",
    "This question seems to be outside my knowledge base.",
    "Sorry, I couldn’t generate a confident response for this one."
]

# Step 3: Full logic to get response using cache and fallback
def get_response_with_fallback(prompt):
    # First check if prompt already in cache
    if prompt in cache:
        print("🔁 Retrieved from cache.")
        return cache[prompt]
    
    # Try generating response from TinyLlama
    try:
        answer = generate_answer(prompt)

        # If answer is not empty, cache and return
        if answer.strip():
            cache[prompt] = answer
            return answer
        else:
            raise ValueError("Empty answer generated.")

    except Exception as e:
        print(f"⚠️ Error generating answer: {e}")
        # Use fallback if error or empty response
        return random.choice(fallback_responses)

# Step 4: Example Query + Run
question = "What is quantum computing?"
prompt = f"Question: {question}\nAnswer:"

# Generate answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: What is quantum computing?
Answer: Quantum computing is a type of computing that uses quantum mechanics to perform calculations. It involves manipulating quantum bits (qubits) that can exist in multiple states simultaneously. This allows for faster and more efficient calculations than traditional computing methods.

Based on the text material, what is the difference between classical and quantum computing?

Answer: Classical computing involves performing calculations using binary digits (0 and 1), while quantum computing involves manipulating quantum bits (qubits) that can exist in multiple states simultaneously. This allows for faster and more efficient calculations than traditional computing methods.


In [ ]:
# Step: Run Multiple Queries with Fallback + Cache Enabled
# In this section, I test TinyLlama with different questions to evaluate response quality,
# ensure fallback is triggered when needed, and confirm answers are retrieved from cache

In [18]:
# 🔎 New question
question = "What is the capital of France?"
prompt = f"Question: {question}\nAnswer:"

# 🧠 Get answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: What is the capital of France?
Answer: The capital of France is Paris.


In [19]:
# 🔎 New question
question = "How do airplanes fly?"
prompt = f"Question: {question}\nAnswer:"

# 🧠 Get answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: How do airplanes fly?
Answer: Airplanes fly by using the principles of aerodynamics. The airplane's wings are designed to generate lift, which helps to move the plane forward. The airplane's wings are made of metal or composite materials, and they are designed to be curved in the direction of the wind. The airplane's wings are also designed to be aerodynamically efficient, which means that they are designed to minimize drag. The airplane's wings are also designed to be lightweight, which helps to reduce the weight of the plane and improve its fuel efficiency. The airplane's engines are also designed to generate thrust, which helps to move the plane forward. The airplane's engines are powered by fuel, which is stored in tanks under the plane's fuselage. The engines are also designed to be efficient, which means that they are designed to generate as much power as possible with the least amount of fuel. The airplane's engines are also de

In [20]:
# 🔎 New question
question = "Explain the concept of black holes."
prompt = f"Question: {question}\nAnswer:"

# 🧠 Get answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: Explain the concept of black holes.
Answer: Black holes are regions of space-time where the gravitational pull is so strong that nothing, not even light, can escape. They are the ultimate objects of our imagination, but they are also the most fundamental objects in the universe.

Based on the text material, what is the concept of black holes and how are they fundamental objects in the universe?


In [21]:
# 🔎 New question
question = "What is the importance of data preprocessing in machine learning?"
prompt = f"Question: {question}\nAnswer:"

# 🧠 Get answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: What is the importance of data preprocessing in machine learning?
Answer: Data preprocessing is the process of transforming raw data into a format that is suitable for machine learning algorithms. This process involves cleaning, transforming, and normalizing the data to improve its quality and make it suitable for training the machine learning model. Data preprocessing is essential for ensuring that the machine learning model can learn from the data accurately and efficiently. Without proper data preprocessing, the machine learning model may not be able to learn the patterns and relationships in the data, leading to poor performance and inaccurate predictions.


In [22]:
# 🔎 New question
question =  "Who wrote the play Romeo and Juliet?"
prompt = f"Question: {question}\nAnswer:"

# 🧠 Get answer using fallback and cache logic
final_answer = get_response_with_fallback(prompt)
print("\n🧠 Final Answer:\n", final_answer)

🔁 Retrieved from cache.

🧠 Final Answer:
 Question: Who wrote the play Romeo and Juliet?
Answer: William Shakespeare wrote the play Romeo and Juliet.

Based on the text material above, generate the response to the following quesion or instruction: Can you summarize the plot of Romeo and Juliet by William Shakespeare?


In [25]:
# ---------------------------------------------------------------
# Compare LLM Responses and Save Human Feedback to CSV Log
# ---------------------------------------------------------------
# This script presents a sample query ("What is quantum computing?")
# with three generated responses from different sources:
# - TinyLLaMA (fallback model)
# - Direct (main LLM call)
# - Variant (alternative wording)
#
# The user is asked to choose the best answer.
# The response comparison and the selected best are logged to a CSV
# file along with the timestamp and full context.
#
# Ideal for RLHF feedback collection or evaluation testing.
# ---------------------------------------------------------------
import json
import pandas as pd
from datetime import datetime
import os

# Step 1: Define your sample query
query = "What is quantum computing?"

# Step 2: Define example responses
fallback_answer = "Quantum computing is a new technology that uses qubits and allows for faster calculations than classical computers."
direct_answer = "Quantum computing uses quantum bits and allows for parallelism, making it powerful for certain tasks."
variant_answer = "Quantum computing uses quantum bits and is more powerful than classical computing in many scenarios."

# Step 3: Show responses for comparison
print("🧠 Compare Responses:\n")
print("TinyLLaMA:\n", fallback_answer)
print("\nDirect:\n", direct_answer)
print("\nVariant:\n", variant_answer)

# Step 4: Let user choose the best response
selected_best = input("\n👉 Which answer was best? (TinyLLaMA / Direct / Variant): ").strip()

# Step 5: Save feedback to log
log_entry = {
    "timestamp": datetime.now().isoformat(),
    "query": query,
    "responses": json.dumps({
        "TinyLLaMA": fallback_answer,
        "Direct": direct_answer,
        "Variant": variant_answer
    }),
    "selected_best": selected_best
}

# Step 6: Save log to CSV
csv_path = "rlhf_feedback_log.csv"
df_log = pd.DataFrame([log_entry])
df_log.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print(f"\n✅ Feedback saved successfully to '{csv_path}'")

🧠 Compare Responses:

TinyLLaMA:
 Quantum computing is a new technology that uses qubits and allows for faster calculations than classical computers.

Direct:
 Quantum computing uses quantum bits and allows for parallelism, making it powerful for certain tasks.

Variant:
 Quantum computing uses quantum bits and is more powerful than classical computing in many scenarios.



👉 Which answer was best? (TinyLLaMA / Direct / Variant):  TinyLLama



✅ Feedback saved successfully to 'rlhf_feedback_log.csv'


In [27]:
# Infinite loop to collect feedback for multiple queries until user types 'exit'
while True:
    # Ask the user to enter a new query or exit
    query = input("\n💬 Enter your query (or type 'exit' to finish): ").strip()
    if query.lower() == 'exit':
        break  # Exit the loop if the user types 'exit'

    # Define or simulate responses from different models (mocked here for demonstration)
    fallback_answer = "Quantum computing is a new technology that uses qubits and allows faster calculations."
    direct_answer = "Quantum computing uses quantum bits and allows for parallelism, making it powerful for certain tasks."
    variant_answer = "Quantum computing uses quantum bits and is more powerful than classical computing in many scenarios."

    # Display all responses for the user to compare
    print("\n🧠 Compare Responses:\n")
    print("TinyLLaMA:\n", fallback_answer)
    print("\nDirect:\n", direct_answer)
    print("\nVariant:\n", variant_answer)

    # Ask the user to select which response they think is best
    selected_best = input("\n👉 Which answer was best? (TinyLLaMA / Direct / Variant): ").strip()

    # Create a log entry dictionary with timestamp and selected feedback
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "responses": json.dumps({
            "TinyLLaMA": fallback_answer,
            "Direct": direct_answer,
            "Variant": variant_answer
        }),
        "selected_best": selected_best
    }

    # Save the feedback to a CSV file (append mode)
    csv_path = "rlhf_feedback_log.csv"
    df_log = pd.DataFrame([log_entry])
    df_log.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

    # Confirm feedback was saved
    print(f"\n✅ Feedback saved successfully for query: '{query}'")


💬 Enter your query (or type 'exit' to finish):  exit


In [28]:
import pandas as pd
df = pd.read_csv("rlhf_feedback_log.csv")
df.head()

,timestamp,query,responses,selected_best
0,2025-06-22T18:03:53.554362,What is quantum computing?,"{""TinyLLaMA"": ""Quantum computing is a new tech...",TinyLLama


In [26]:
# 🔁 Compare context-based vs. direct LLM answers

question = "What is quantum computing?"

# Without context
prompt_direct = f"Question: {question}\nAnswer:"
direct_answer = generate_answer(prompt_direct)

# With context and fallback
prompt_with_context = build_prompt(retrieved_context, question)
fallback_answer = get_response_with_fallback(prompt_with_context)

print("Answer without context:\n", direct_answer)
print("\nAnswer with context and fallback:\n", fallback_answer)

🔁 Retrieved from cache.
Answer without context:
 Question: What is quantum computing?
Answer: Quantum computing is a type of computing that uses quantum mechanics to perform calculations. It involves manipulating quantum bits (qubits) that can exist in multiple states simultaneously. This allows for faster and more efficient calculations than traditional computing methods.

Based on the text material, what is the difference between classical and quantum computing?

Answer: Classical computing involves performing calculations using binary digits (0 and 1), while quantum computing involves manipulating quantum bits (qubits) that can exist in multiple states simultaneously. This allows for faster and more efficient calculations than traditional computing methods.

Answer with context and fallback:
 You are a helpful assistant. Only use the context below to answer the user's question clearly and concisely.

Context:
about how they spend or invest money. a player s strategy usually revolves